## 1 安装依赖

In [ ]:
# @title 1. 环境准备与项目克隆
!git clone https://github.com/ctz168/stdpbrain.git
%cd stdpbrain
!pip install -r requirements.txt --break-system-packages
!pip install torch transformers>=5.3.0 huggingface_hub accelerate --break-system-packages
!pip install python-telegram-bot aiohttp pydantic python-dotenv loguru tqdm colorama --break-system-packages
!pip install flash-linear-attention causal-conv1d --break-system-packages
print("✅ Python 依赖安装完成")

## 2 下载模型

In [ ]:
# 国外用hf
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
!python download_model.py
print("✅ 模型下载完成")

In [ ]:
#国内用Modelscope
!modelscope download Qwen/Qwen3.5-0.8B --local_dir ./models/Qwen3.5-0.8B

## 3 验证窄带宽注意力补丁

In [ ]:
# @title 验证窄带宽注意力补丁
# @markdown 检查 Monkey Patch 是否正确应用到 Qwen 注意力层

import sys
sys.path.insert(0, '/content/stdpbrain')

# 导入补丁模块
from core.qwen_narrow_band_patch import get_memory_anchor_store, patch_qwen_attention

# 检查全局存储
anchor_store = get_memory_anchor_store()
print(f"✅ 记忆锚点存储已创建")
print(f"   - 最大锚点数: {anchor_store.max_anchors}")
print(f"   - 启用状态: {anchor_store.enabled}")

# 应用补丁（模型加载时会自动应用，这里手动验证）
try:
    success = patch_qwen_attention()
    if success:
        print(f"\n✅ 窄带宽注意力补丁已成功应用")
        print(f"   - 类人脑稀疏注意力机制: 启用")
        print(f"   - 海马体记忆锚点注入: 启用")
        print(f"   - 复杂度: O((n+k)·d)，k=3-5 (常数级)")
        print(f"   - 预期性能提升: 20-50%")
    else:
        print(f"\n⚠️ 补丁应用失败，将在模型加载时重试")
except Exception as e:
    print(f"\n⚠️ 补丁验证跳过: {e}")
    print(f"   补丁将在模型加载时自动应用")

## 4 配置并运行 Bot

In [ ]:
# @title 运行 Bot
# @markdown 点击运行后，Bot 将持续运行直到单元格被中断

import os
import sys

# 切换到项目目录
os.chdir('/content/stdpbrain')
sys.path.insert(0, '/content/stdpbrain')

!python main.py --mode telegram --telegram-token 7983263905:AAFsMuGRdZzWv7KfUaAkJocu0l7LsHrScuc